# Differential gene expression between individuals who are non diabetic or individuals with type 2 diabetes

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_deseq
# python -m ipykernel install --user --name scrna_cartography_dreampy --display-name "deseq"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

import matplotlib.pyplot as plt  # Plotting interface

# dream
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

#### Set up

In [2]:
# Paths
base_dir = str(here('data/differential_genes_across_disease/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
diffg_dir = os.path.join(base_dir, 'nd_t2d') 
tmp_dir = os.path.join(base_dir, 'tmp') 

anndata_dir = str(here('data/anndata/'))

mi.create_directories(os.path.join(base_dir, 'nd_t2d'))
mi.create_directories(os.path.join(plot_dir))
mi.create_directories(os.path.join(files_dir))
mi.create_directories(os.path.join(diffg_dir))
mi.create_directories(os.path.join(base_dir, 'tmp'))

/work/islet_cartography_scrna/data/differential_genes_across_disease/nd_t2d Directory already exists!
/work/islet_cartography_scrna/data/differential_genes_across_disease/plot Directory already exists!
/work/islet_cartography_scrna/data/differential_genes_across_disease/files Directory already exists!
/work/islet_cartography_scrna/data/differential_genes_across_disease/nd_t2d Directory already exists!
/work/islet_cartography_scrna/data/differential_genes_across_disease/tmp Directory already exists!


#### Load

In [3]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))
duplicated_donors = pd.read_csv(str(here('data/duplicated_donors.csv')), sep = "\t")

#### Diff genes per dataset

#### Remove duplicated donors

In [4]:
# Setup -----------------------------------------------------------------------------
anno_key   = "cell_type"
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
disease_key = "disease_harmonized"
target_celltypes = adata.obs[anno_key].unique()
inference = DefaultInference(n_cpus=-1)

In [5]:
# Create combined unique string keys
remove_keys = (
    duplicated_donors["ic_id_donor_overall"].astype(str)
    + "__"
    + duplicated_donors["ic_id_dataset"].astype(str)
)

obs_keys = (
    adata.obs["ic_id_donor_overall"].astype(str)
    + "__"
    + adata.obs["ic_id_dataset"].astype(str)
)

# Build mask and slice
mask_remove = obs_keys.isin(remove_keys)
adata = adata[~mask_remove].copy()

remaining_multi = adata.obs.groupby(donor_key)[dataset_key].nunique()
n_remaining_multi = (remaining_multi > 1).sum()
assert n_remaining_multi == 0, f"Error: {n_remaining_multi} donors still appear in multiple datasets!"
print("Filtering successful: all donors now in single dataset")
print("Number of donors: ", len(remaining_multi))
print(f"Removed {(mask_remove).sum()} observations")

Filtering successful: all donors now in single dataset
Number of donors:  270
Removed 1118 observations
